In [1]:
import pandas as pd
import pickle
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn_quantile import RandomForestQuantileRegressor
from tqdm.auto import tqdm

import sys
sys.path.append('../../../TaskExecutionTimeMining/')
from quantile_regression import QuantileRegression


In [2]:
with open('../../transformed_event_logs/BPIC_2017_all_train.pickle', 'rb') as f:
    train_data = pickle.load(f)

In [3]:
train_data

,Action_start,org:resource_start,concept:name,EventOrigin_start,EventID_start,lifecycle:transition_start,time:timestamp_start,case:LoanGoal_start,case:ApplicationType_start,case:concept:name,...,intercase_n_3__W_Validate application__suspend_W_Shortened completion __resume_W_Validate application__ate_abort,intercase_n_3__W_Validate application__suspend_W_Shortened completion __resume_W_Validate application__resume,intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Assess potential fraud__schedule,intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Call incomplete files__schedule,intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Validate application__schedule,intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Assess potential fraud__schedule,intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Call incomplete files__schedule,intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__complete,intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__start,intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__suspend
0,Created,User_112,W_Validate application__schedule,Workflow,Workitem_100000116,schedule,2016-03-02 13:25:52.388000+00:00,Home improvement,Limit raise,Application_1650545424,...,0,0,0,0,0,0,0,0,0,9
1,Released,User_63,W_Call after offers__suspend,Workflow,Workitem_1000010198,suspend,2016-08-20 12:19:22.434000+00:00,Home improvement,New credit,Application_1930272371,...,0,0,0,0,0,0,0,0,0,10
2,Obtained,User_80,W_Call after offers__start,Workflow,Workitem_1000013868,start,2016-10-18 08:54:52.604000+00:00,Car,New credit,Application_1804686886,...,0,0,0,0,0,0,0,0,0,6
3,Released,User_30,W_Call incomplete files__suspend,Workflow,Workitem_1000014801,suspend,2016-11-30 16:55:49.483000+00:00,Not speficied,New credit,Application_655052891,...,0,0,0,0,0,0,0,0,0,12
4,Obtained,User_114,W_Validate application__start,Workflow,Workitem_1000015916,start,2016-01-19 12:58:12.261000+00:00,Car,New credit,Application_704665572,...,0,0,0,0,0,0,0,0,0,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
619773,Released,User_112,W_Validate application__suspend,Workflow,Workitem_999946421,suspend,2016-03-07 11:20:38.606000+00:00,Home improvement,New credit,Application_1748541113,...,0,0,0,0,0,0,0,0,0,24
619774,Created,User_116,W_Validate application__schedule,Workflow,Workitem_999950446,schedule,2016-07-15 12:55:25.889000+00:00,Existing loan takeover,New credit,Application_519919221,...,0,0,0,0,0,0,0,0,0,51
619777,Released,User_19,W_Call after offers__suspend,Workflow,Workitem_999988201,suspend,2016-10-04 18:43:06.137000+00:00,Home improvement,New credit,Application_1551324804,...,0,0,0,0,0,0,0,0,0,16
619779,Released,User_51,W_Call after offers__suspend,Workflow,Workitem_999991648,suspend,2016-12-01 20:02:50.875000+00:00,Not speficied,New credit,Application_1000610355,...,0,0,0,0,0,0,0,0,0,12


In [4]:
list(train_data.columns)

['Action_start',
 'org:resource_start',
 'concept:name',
 'EventOrigin_start',
 'EventID_start',
 'lifecycle:transition_start',
 'time:timestamp_start',
 'case:LoanGoal_start',
 'case:ApplicationType_start',
 'case:concept:name',
 'case:RequestedAmount_start',
 'FirstWithdrawalAmount_start',
 'NumberOfTerms_start',
 'Accepted_start',
 'MonthlyCost_start',
 'Selected_start',
 'CreditScore_start',
 'OfferedAmount_start',
 'OfferID_start',
 'Action_complete',
 'org:resource_complete',
 'EventOrigin_complete',
 'EventID_complete',
 'lifecycle:transition_complete',
 'time:timestamp_complete',
 'case:LoanGoal_complete',
 'case:ApplicationType_complete',
 'case:RequestedAmount_complete',
 'FirstWithdrawalAmount_complete',
 'NumberOfTerms_complete',
 'Accepted_complete',
 'MonthlyCost_complete',
 'Selected_complete',
 'CreditScore_complete',
 'OfferedAmount_complete',
 'OfferID_complete',
 'duration',
 'duration_seconds',
 'duration_ms',
 'duration_hours',
 'seconds_in_day',
 'day_of_week',
 '

In [5]:
activity_count = [
 'W_Assess potential fraud__ate_abort',
 'W_Assess potential fraud__complete',
 'W_Assess potential fraud__resume',
 'W_Assess potential fraud__schedule',
 'W_Assess potential fraud__start',
 'W_Assess potential fraud__suspend',
 'W_Assess potential fraud__withdraw',
 'W_Call after offers__ate_abort',
 'W_Call after offers__complete',
 'W_Call after offers__resume',
 'W_Call after offers__schedule',
 'W_Call after offers__start',
 'W_Call after offers__suspend',
 'W_Call after offers__withdraw',
 'W_Call incomplete files__ate_abort',
 'W_Call incomplete files__complete',
 'W_Call incomplete files__resume',
 'W_Call incomplete files__schedule',
 'W_Call incomplete files__start',
 'W_Call incomplete files__suspend',
 'W_Complete application__ate_abort',
 'W_Complete application__complete',
 'W_Complete application__resume',
 'W_Complete application__schedule',
 'W_Complete application__start',
 'W_Complete application__suspend',
 'W_Handle leads__complete',
 'W_Handle leads__resume',
 'W_Handle leads__schedule',
 'W_Handle leads__start',
 'W_Handle leads__suspend',
 'W_Handle leads__withdraw',
 'W_Shortened completion __resume',
 'W_Shortened completion __schedule',
 'W_Shortened completion __start',
 'W_Shortened completion __suspend',
 'W_Validate application__ate_abort',
 'W_Validate application__complete',
 'W_Validate application__resume',
 'W_Validate application__schedule',
 'W_Validate application__start',
 'W_Validate application__suspend',
 ]

resource_count = [
'User_1',
 'User_10',
 'User_100',
 'User_101',
 'User_102',
 'User_103',
 'User_104',
 'User_105',
 'User_106',
 'User_107',
 'User_108',
 'User_109',
 'User_11',
 'User_110',
 'User_111',
 'User_112',
 'User_113',
 'User_114',
 'User_115',
 'User_116',
 'User_117',
 'User_118',
 'User_119',
 'User_12',
 'User_120',
 'User_121',
 'User_122',
 'User_123',
 'User_124',
 'User_125',
 'User_126',
 'User_127',
 'User_128',
 'User_129',
 'User_13',
 'User_130',
 'User_131',
 'User_132',
 'User_133',
 'User_134',
 'User_135',
 'User_136',
 'User_137',
 'User_138',
 'User_139',
 'User_14',
 'User_140',
 'User_141',
 'User_142',
 'User_143',
 'User_144',
 'User_145',
 'User_146',
 'User_147',
 'User_148',
 'User_149',
 'User_15',
 'User_16',
 'User_17',
 'User_18',
 'User_19',
 'User_2',
 'User_20',
 'User_21',
 'User_22',
 'User_23',
 'User_24',
 'User_25',
 'User_26',
 'User_27',
 'User_28',
 'User_29',
 'User_3',
 'User_30',
 'User_31',
 'User_32',
 'User_33',
 'User_34',
 'User_35',
 'User_36',
 'User_37',
 'User_38',
 'User_39',
 'User_4',
 'User_40',
 'User_41',
 'User_42',
 'User_43',
 'User_44',
 'User_45',
 'User_46',
 'User_47',
 'User_48',
 'User_49',
 'User_5',
 'User_50',
 'User_51',
 'User_52',
 'User_53',
 'User_54',
 'User_55',
 'User_56',
 'User_57',
 'User_58',
 'User_59',
 'User_6',
 'User_60',
 'User_61',
 'User_62',
 'User_63',
 'User_64',
 'User_65',
 'User_66',
 'User_67',
 'User_68',
 'User_69',
 'User_7',
 'User_70',
 'User_71',
 'User_72',
 'User_73',
 'User_74',
 'User_75',
 'User_76',
 'User_77',
 'User_78',
 'User_79',
 'User_8',
 'User_80',
 'User_81',
 'User_82',
 'User_83',
 'User_84',
 'User_85',
 'User_86',
 'User_87',
 'User_88',
 'User_89',
 'User_9',
 'User_90',
 'User_91',
 'User_92',
 'User_93',
 'User_94',
 'User_95',
 'User_96',
 'User_97',
 'User_98',
 'User_99',
]

ii1 = [
    'intercase_n_1__W_Assess potential fraud__ate_abort',
 'intercase_n_1__W_Assess potential fraud__complete',
 'intercase_n_1__W_Assess potential fraud__resume',
 'intercase_n_1__W_Assess potential fraud__schedule',
 'intercase_n_1__W_Assess potential fraud__start',
 'intercase_n_1__W_Assess potential fraud__suspend',
 'intercase_n_1__W_Assess potential fraud__withdraw',
 'intercase_n_1__W_Call after offers__ate_abort',
 'intercase_n_1__W_Call after offers__complete',
 'intercase_n_1__W_Call after offers__resume',
 'intercase_n_1__W_Call after offers__schedule',
 'intercase_n_1__W_Call after offers__start',
 'intercase_n_1__W_Call after offers__suspend',
 'intercase_n_1__W_Call after offers__withdraw',
 'intercase_n_1__W_Call incomplete files__ate_abort',
 'intercase_n_1__W_Call incomplete files__complete',
 'intercase_n_1__W_Call incomplete files__resume',
 'intercase_n_1__W_Call incomplete files__schedule',
 'intercase_n_1__W_Call incomplete files__start',
 'intercase_n_1__W_Call incomplete files__suspend',
 'intercase_n_1__W_Complete application__ate_abort',
 'intercase_n_1__W_Complete application__complete',
 'intercase_n_1__W_Complete application__resume',
 'intercase_n_1__W_Complete application__schedule',
 'intercase_n_1__W_Complete application__start',
 'intercase_n_1__W_Complete application__suspend',
 'intercase_n_1__W_Handle leads__complete',
 'intercase_n_1__W_Handle leads__resume',
 'intercase_n_1__W_Handle leads__schedule',
 'intercase_n_1__W_Handle leads__start',
 'intercase_n_1__W_Handle leads__suspend',
 'intercase_n_1__W_Handle leads__withdraw',
 'intercase_n_1__W_Shortened completion __resume',
 'intercase_n_1__W_Shortened completion __schedule',
 'intercase_n_1__W_Shortened completion __start',
 'intercase_n_1__W_Shortened completion __suspend',
 'intercase_n_1__W_Validate application__ate_abort',
 'intercase_n_1__W_Validate application__complete',
 'intercase_n_1__W_Validate application__resume',
 'intercase_n_1__W_Validate application__schedule',
 'intercase_n_1__W_Validate application__start',
 'intercase_n_1__W_Validate application__suspend',
]

ii3 = [
    'intercase_n_3__W_Assess potential fraud__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Assess potential fraud__ate_abort_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Assess potential fraud__complete_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Assess potential fraud__complete_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Assess potential fraud__complete_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__complete_W_Validate application__schedule',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__start_W_Assess potential fraud__suspend',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__start_W_Call after offers__schedule',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__start_W_Handle leads__schedule',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__start_W_Validate application__schedule',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__suspend_W_Assess potential fraud__resume',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__suspend_W_Complete application__schedule',
 'intercase_n_3__W_Assess potential fraud__resume_W_Assess potential fraud__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Assess potential fraud__resume_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Assess potential fraud__resume_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Assess potential fraud__resume_W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Assess potential fraud__resume_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__start_W_Assess potential fraud__complete',
 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__start_W_Assess potential fraud__suspend',
 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__start_W_Call after offers__schedule',
 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__start_W_Handle leads__schedule',
 'intercase_n_3__W_Assess potential fraud__schedule_W_Assess potential fraud__withdraw_W_Handle leads__schedule',
 'intercase_n_3__W_Assess potential fraud__schedule_W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__complete_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__complete_W_Complete application__schedule',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Assess potential fraud__ate_abort',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Assess potential fraud__resume',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Call after offers__schedule',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Complete application__schedule',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Handle leads__schedule',
 'intercase_n_3__W_Assess potential fraud__start_W_Assess potential fraud__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Assess potential fraud__start_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Assess potential fraud__start_W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Assess potential fraud__start_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__ate_abort_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__ate_abort_W_Validate application__schedule',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Assess potential fraud__complete',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Assess potential fraud__start',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Assess potential fraud__suspend',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Call after offers__schedule',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Complete application__schedule',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Handle leads__schedule',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Assess potential fraud__resume_W_Validate application__schedule',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Assess potential fraud__suspend_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Assess potential fraud__withdraw_W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Call after offers__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call after offers__ate_abort_W_Call after offers__schedule_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call after offers__ate_abort_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Call after offers__ate_abort_W_Call after offers__schedule_W_Call after offers__withdraw',
 'intercase_n_3__W_Call after offers__ate_abort_W_Call after offers__schedule_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__ate_abort_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call after offers__complete_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call after offers__resume_W_Call after offers__start_W_Call after offers__suspend',
 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Call after offers__ate_abort',
 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Call after offers__resume',
 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Shortened completion __resume',
 'intercase_n_3__W_Call after offers__resume_W_Call after offers__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__resume_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call after offers__schedule_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Call after offers__complete',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Call after offers__suspend',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Shortened completion __schedule',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Shortened completion __suspend',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__start_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__withdraw_W_Call after offers__schedule',
 'intercase_n_3__W_Call after offers__schedule_W_Call after offers__withdraw_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__schedule_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call after offers__start_W_Call after offers__complete_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Call after offers__ate_abort',
 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Call after offers__resume',
 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Shortened completion __schedule',
 'intercase_n_3__W_Call after offers__start_W_Call after offers__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__start_W_Shortened completion __schedule_W_Shortened completion __start',
 'intercase_n_3__W_Call after offers__start_W_Shortened completion __suspend_W_Call after offers__suspend',
 'intercase_n_3__W_Call after offers__start_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call after offers__suspend_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__ate_abort_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__ate_abort_W_Call after offers__schedule',
 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__ate_abort_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__resume_W_Call after offers__start',
 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__resume_W_Call after offers__suspend',
 'intercase_n_3__W_Call after offers__suspend_W_Call after offers__resume_W_Validate application__schedule',
 'intercase_n_3__W_Call after offers__suspend_W_Shortened completion __schedule_W_Shortened completion __start',
 'intercase_n_3__W_Call after offers__suspend_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call after offers__withdraw_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Call after offers__withdraw_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call incomplete files__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call incomplete files__ate_abort_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Call incomplete files__ate_abort_W_Call incomplete files__schedule_W_Call incomplete files__start',
 'intercase_n_3__W_Call incomplete files__ate_abort_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call incomplete files__complete_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call incomplete files__complete_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call incomplete files__resume_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__complete_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__start_W_Call incomplete files__complete',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__start_W_Call incomplete files__suspend',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__start_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Call after offers__schedule',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Call incomplete files__ate_abort',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Call incomplete files__resume',
 'intercase_n_3__W_Call incomplete files__resume_W_Call incomplete files__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__resume_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call incomplete files__schedule_W_Call incomplete files__start_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call incomplete files__schedule_W_Call incomplete files__start_W_Call incomplete files__complete',
 'intercase_n_3__W_Call incomplete files__schedule_W_Call incomplete files__start_W_Call incomplete files__suspend',
 'intercase_n_3__W_Call incomplete files__schedule_W_Call incomplete files__start_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__start_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__complete_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__complete_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__suspend_W_Call incomplete files__ate_abort',
 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__suspend_W_Call incomplete files__resume',
 'intercase_n_3__W_Call incomplete files__start_W_Call incomplete files__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__start_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Call incomplete files__suspend_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__ate_abort_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__ate_abort_W_Call after offers__schedule',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__ate_abort_W_Call incomplete files__schedule',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__ate_abort_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Call incomplete files__complete',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Call incomplete files__start',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Call incomplete files__suspend',
 'intercase_n_3__W_Call incomplete files__suspend_W_Call incomplete files__resume_W_Validate application__schedule',
 'intercase_n_3__W_Call incomplete files__suspend_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Complete application__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Complete application__ate_abort_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Complete application__complete_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Complete application__complete_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Complete application__resume_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Complete application__resume_W_Complete application__start_W_Call after offers__schedule',
 'intercase_n_3__W_Complete application__resume_W_Complete application__start_W_Complete application__suspend',
 'intercase_n_3__W_Complete application__resume_W_Complete application__suspend_W_Call after offers__schedule',
 'intercase_n_3__W_Complete application__resume_W_Complete application__suspend_W_Complete application__ate_abort',
 'intercase_n_3__W_Complete application__resume_W_Complete application__suspend_W_Complete application__resume',
 'intercase_n_3__W_Complete application__resume_W_Shortened completion __schedule_W_Shortened completion __start',
 'intercase_n_3__W_Complete application__schedule',
 'intercase_n_3__W_Complete application__schedule_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Complete application__schedule_W_Complete application__start_W_Call after offers__schedule',
 'intercase_n_3__W_Complete application__schedule_W_Complete application__start_W_Complete application__complete',
 'intercase_n_3__W_Complete application__schedule_W_Complete application__start_W_Complete application__suspend',
 'intercase_n_3__W_Complete application__schedule_W_Complete application__start_W_Shortened completion __schedule',
 'intercase_n_3__W_Complete application__schedule_W_Shortened completion __schedule_W_Shortened completion __start',
 'intercase_n_3__W_Complete application__start_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Complete application__start_W_Complete application__complete_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Complete application__start_W_Complete application__complete_W_Complete application__schedule',
 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Call after offers__schedule',
 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Complete application__ate_abort',
 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Complete application__resume',
 'intercase_n_3__W_Complete application__start_W_Complete application__suspend_W_Shortened completion __schedule',
 'intercase_n_3__W_Complete application__start_W_Shortened completion __schedule_W_Shortened completion __start',
 'intercase_n_3__W_Complete application__suspend_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Complete application__suspend_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Complete application__suspend_W_Complete application__ate_abort_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Complete application__suspend_W_Complete application__ate_abort_W_Complete application__schedule',
 'intercase_n_3__W_Complete application__suspend_W_Complete application__resume_W_Call after offers__schedule',
 'intercase_n_3__W_Complete application__suspend_W_Complete application__resume_W_Complete application__start',
 'intercase_n_3__W_Complete application__suspend_W_Complete application__resume_W_Complete application__suspend',
 'intercase_n_3__W_Complete application__suspend_W_Complete application__resume_W_Shortened completion __schedule',
 'intercase_n_3__W_Complete application__suspend_W_Shortened completion __schedule_W_Shortened completion __start',
 'intercase_n_3__W_Handle leads__complete_W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Handle leads__resume_W_Complete application__schedule_W_Call after offers__schedule',
 'intercase_n_3__W_Handle leads__resume_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Handle leads__resume_W_Handle leads__complete_W_Handle leads__schedule',
 'intercase_n_3__W_Handle leads__resume_W_Handle leads__start_W_Complete application__schedule',
 'intercase_n_3__W_Handle leads__resume_W_Handle leads__start_W_Handle leads__suspend',
 'intercase_n_3__W_Handle leads__resume_W_Handle leads__suspend_W_Complete application__schedule',
 'intercase_n_3__W_Handle leads__resume_W_Handle leads__suspend_W_Handle leads__resume',
 'intercase_n_3__W_Handle leads__schedule',
 'intercase_n_3__W_Handle leads__schedule_W_Complete application__schedule',
 'intercase_n_3__W_Handle leads__schedule_W_Complete application__schedule_W_Call after offers__schedule',
 'intercase_n_3__W_Handle leads__schedule_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Handle leads__schedule_W_Complete application__schedule_W_Shortened completion __schedule',
 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__start',
 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__start_W_Complete application__schedule',
 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__start_W_Handle leads__suspend',
 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__withdraw',
 'intercase_n_3__W_Handle leads__schedule_W_Handle leads__withdraw_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Handle leads__start_W_Complete application__schedule_W_Call after offers__schedule',
 'intercase_n_3__W_Handle leads__start_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Handle leads__start_W_Handle leads__suspend_W_Complete application__schedule',
 'intercase_n_3__W_Handle leads__start_W_Handle leads__suspend_W_Handle leads__resume',
 'intercase_n_3__W_Handle leads__suspend_W_Complete application__schedule_W_Call after offers__schedule',
 'intercase_n_3__W_Handle leads__suspend_W_Complete application__schedule_W_Complete application__start',
 'intercase_n_3__W_Handle leads__suspend_W_Handle leads__resume_W_Complete application__schedule',
 'intercase_n_3__W_Handle leads__suspend_W_Handle leads__resume_W_Handle leads__complete',
 'intercase_n_3__W_Handle leads__suspend_W_Handle leads__resume_W_Handle leads__start',
 'intercase_n_3__W_Handle leads__suspend_W_Handle leads__resume_W_Handle leads__suspend',
 'intercase_n_3__W_Handle leads__withdraw_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Handle leads__withdraw_W_Assess potential fraud__schedule_W_Assess potential fraud__withdraw',
 'intercase_n_3__W_Handle leads__withdraw_W_Assess potential fraud__schedule_W_Handle leads__schedule',
 'intercase_n_3__W_Shortened completion __resume_W_Shortened completion __suspend_W_Shortened completion __resume',
 'intercase_n_3__W_Shortened completion __resume_W_Validate application__ate_abort_W_Call incomplete files__schedule',
 'intercase_n_3__W_Shortened completion __resume_W_Validate application__resume_W_Validate application__suspend',
 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Call after offers__resume',
 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Call after offers__schedule',
 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Call after offers__suspend',
 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Complete application__suspend',
 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Shortened completion __suspend',
 'intercase_n_3__W_Shortened completion __schedule_W_Shortened completion __start_W_Validate application__schedule',
 'intercase_n_3__W_Shortened completion __start_W_Call after offers__resume_W_Call after offers__suspend',
 'intercase_n_3__W_Shortened completion __start_W_Call after offers__schedule_W_Call after offers__start',
 'intercase_n_3__W_Shortened completion __start_W_Call after offers__suspend_W_Call after offers__resume',
 'intercase_n_3__W_Shortened completion __start_W_Call after offers__suspend_W_Validate application__schedule',
 'intercase_n_3__W_Shortened completion __start_W_Complete application__suspend_W_Call after offers__schedule',
 'intercase_n_3__W_Shortened completion __start_W_Complete application__suspend_W_Complete application__resume',
 'intercase_n_3__W_Shortened completion __start_W_Shortened completion __suspend_W_Call after offers__resume',
 'intercase_n_3__W_Shortened completion __start_W_Shortened completion __suspend_W_Complete application__start',
 'intercase_n_3__W_Shortened completion __start_W_Shortened completion __suspend_W_Complete application__suspend',
 'intercase_n_3__W_Shortened completion __start_W_Shortened completion __suspend_W_Shortened completion __resume',
 'intercase_n_3__W_Shortened completion __start_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Shortened completion __suspend_W_Call after offers__resume_W_Call after offers__suspend',
 'intercase_n_3__W_Shortened completion __suspend_W_Call after offers__suspend_W_Call after offers__resume',
 'intercase_n_3__W_Shortened completion __suspend_W_Complete application__start_W_Complete application__suspend',
 'intercase_n_3__W_Shortened completion __suspend_W_Complete application__suspend_W_Complete application__resume',
 'intercase_n_3__W_Shortened completion __suspend_W_Shortened completion __resume_W_Shortened completion __suspend',
 'intercase_n_3__W_Validate application__ate_abort_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Validate application__ate_abort_W_Call incomplete files__schedule_W_Call incomplete files__start',
 'intercase_n_3__W_Validate application__ate_abort_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Validate application__complete_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Validate application__complete_W_Call incomplete files__schedule_W_Call incomplete files__start',
 'intercase_n_3__W_Validate application__complete_W_Validate application__schedule_W_Validate application__start',
 'intercase_n_3__W_Validate application__resume_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Validate application__resume_W_Call incomplete files__schedule_W_Call incomplete files__start',
 'intercase_n_3__W_Validate application__resume_W_Validate application__complete_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__resume_W_Validate application__complete_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__resume_W_Validate application__start_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__resume_W_Validate application__start_W_Validate application__complete',
 'intercase_n_3__W_Validate application__resume_W_Validate application__start_W_Validate application__suspend',
 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Shortened completion __resume',
 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Validate application__ate_abort',
 'intercase_n_3__W_Validate application__resume_W_Validate application__suspend_W_Validate application__resume',
 'intercase_n_3__W_Validate application__schedule_W_Validate application__start_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__schedule_W_Validate application__start_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__schedule_W_Validate application__start_W_Validate application__complete',
 'intercase_n_3__W_Validate application__schedule_W_Validate application__start_W_Validate application__suspend',
 'intercase_n_3__W_Validate application__start_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Validate application__start_W_Call incomplete files__schedule_W_Call incomplete files__start',
 'intercase_n_3__W_Validate application__start_W_Validate application__complete_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__start_W_Validate application__complete_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__start_W_Validate application__complete_W_Validate application__schedule',
 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Shortened completion __resume',
 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Validate application__ate_abort',
 'intercase_n_3__W_Validate application__start_W_Validate application__suspend_W_Validate application__resume',
 'intercase_n_3__W_Validate application__suspend_W_Assess potential fraud__schedule_W_Assess potential fraud__start',
 'intercase_n_3__W_Validate application__suspend_W_Call incomplete files__schedule_W_Call incomplete files__start',
 'intercase_n_3__W_Validate application__suspend_W_Shortened completion __resume_W_Validate application__ate_abort',
 'intercase_n_3__W_Validate application__suspend_W_Shortened completion __resume_W_Validate application__resume',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__ate_abort_W_Validate application__schedule',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Assess potential fraud__schedule',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Call incomplete files__schedule',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__complete',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__start',
 'intercase_n_3__W_Validate application__suspend_W_Validate application__resume_W_Validate application__suspend'
]

feature_combinations = {
        'A' : ['concept:name'],
        'R' : ['org:resource_start'],
        'AR' : ['concept:name', 'org:resource_start'],
        'ARS' : ['concept:name', 'org:resource_start', 'seconds_in_day'],
        #'ASAC' : ['concept:name', 'seconds_in_day'] + activity_count,
        #'RSRC' : ['org:resource_start', 'seconds_in_day'] + resource_count,
        'ARSAC' : ['concept:name', 'org:resource_start', 'seconds_in_day'] + activity_count,
        'ARSRC' : ['concept:name', 'org:resource_start', 'seconds_in_day'] + resource_count,
        'ARSACRC' : ['concept:name', 'org:resource_start', 'seconds_in_day'] + activity_count + resource_count,
        #'ARACRC' : ['concept:name', 'org:resource_start'] + activity_count + resource_count,
        'ARSD' : ['concept:name', 'org:resource_start', 'seconds_in_day', 'day_of_week'],
        'ARSDACRC' : ['concept:name', 'org:resource_start', 'seconds_in_day', 'day_of_week'] + activity_count + resource_count,
        'ARSDII1' : ['concept:name', 'org:resource_start', 'seconds_in_day', 'day_of_week'] + ii1,
        'ARSDII3' : ['concept:name', 'org:resource_start', 'seconds_in_day', 'day_of_week'] + ii3,
        'ARSDACRCII1' : ['concept:name', 'org:resource_start', 'seconds_in_day', 'day_of_week'] + activity_count + resource_count + ii1,
        'ARSDACRCII3' : ['concept:name', 'org:resource_start', 'seconds_in_day', 'day_of_week'] + activity_count + resource_count + ii3
}

In [6]:
quantile_regression_models = {}
for k, v in tqdm(feature_combinations.items()):
    qrm = QuantileRegression(train_data, v)
    qrm.fit()
    quantile_regression_models[k] = qrm

  0%|          | 0/13 [00:00<?, ?it/s]

/home/LordKunkler/TaskExecutionTimeMining/src/notebooks/evaluate_quantile_regression/BPIC_2017/../../../TaskExecutionTimeMining/quantile_regression.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  encoded_data[col] = codes
/home/LordKunkler/TaskExecutionTimeMining/src/notebooks/evaluate_quantile_regression/BPIC_2017/../../../TaskExecutionTimeMining/quantile_regression.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  encoded_data[col] = codes
/home/LordKunkler/TaskExecutionTimeMining/src/notebooks/evaluate_quantile_regres

In [7]:
out_path = './quantile_regression_models.pkl'
with open(out_path, 'wb') as out_file:
    pickle.dump(quantile_regression_models, out_file, protocol=pickle.HIGHEST_PROTOCOL)